# Lógica y resolución: razonar con reglas, no con datos

**Facsímil 12 · Razonamiento simbólico** — capítulo 1
(lógica proposicional, formas normales, resolución, DPLL y unificación de primer orden).

Mucho antes de que la IA aprendiera de los datos, la IA *deducía*. Le dabas hechos y reglas, y
sacaba conclusiones que estaban garantizadas: si las premisas son verdaderas, la conclusión también
lo es, sin excepciones y sin probabilidades. Esa idea —razonar manipulando símbolos— sostiene los
verificadores que comprueban que un chip o un programa no tienen un fallo, los planificadores, las
bases de conocimiento y los motores que resuelven millones de restricciones por segundo. Y vuelve a
ser actual: a un modelo de lenguaje que alucina le viene de maravilla un motor lógico al lado que
*compruebe* lo que afirma.

En este cuaderno construyes ese motor desde cero, solo con Python. Escribes fórmulas, calculas sus
tablas de verdad, las pasas a cláusulas, e implementas el **algoritmo de resolución**: la regla con
la que una máquina demuestra un teorema buscando una contradicción. Después escribes un mini-DPLL
(el corazón de los resolutores SAT modernos) y un algoritmo de **unificación** que te lleva hasta un
mini-Prolog capaz de responder consultas encadenando reglas. Sin magia: hay símbolos, unas pocas
reglas de manipulación y una búsqueda.

### Qué vas a aprender
- A representar **fórmulas lógicas** como estructuras de datos y a evaluarlas con una **tabla de
  verdad**, decidiendo si una fórmula es válida, satisfacible o una contradicción.
- A comprobar **equivalencias** lógicas (las leyes de De Morgan, la implicación) sin fiarte de la
  intuición: comprobándolas en *todas* las asignaciones.
- A convertir una fórmula a **forma normal conjuntiva** (CNF) y a representar las cláusulas como
  conjuntos de literales.
- El **algoritmo de resolución**: el resolvente de dos cláusulas y un demostrador por **refutación**
  que prueba un argumento derivando la cláusula vacía.
- **DPLL**: propagación unitaria más backtracking para decidir SAT/UNSAT, y por qué mira
  muchísimo menos que una tabla de verdad (el puente con el Facsímil 2).
- **Unificación** de primer orden con *occurs-check*, y un mini-Prolog que responde una consulta
  encadenando cláusulas de Horn, con su traza.

### Cómo se usa este cuaderno
Las celdas vienen **sin ejecutar**: pulsa ▶ (o `Mayús+Enter`) en cada una, de arriba abajo, y verás
nacer cada tabla, cada demostración y cada gráfico. Lee el texto antes de ejecutar; la gracia no es
que «salga», sino entender *por qué* sale.

### Cuánto cuesta
Unos 25 minutos con calma. Python puro con NumPy y Matplotlib; CPU, sin GPU ni cuentas.

> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

## 1. El lenguaje: fórmulas de la lógica proposicional

Empecemos por lo más simple. Una **proposición** es una afirmación que es verdadera o falsa: «llueve»,
«el sensor da alarma», «el número es par». Las nombramos con letras: $p$, $q$, $r$. Con ellas y unos
pocos **conectivos** construimos fórmulas más grandes:

- $\neg p$ — **no** $p$ (negación).
- $p \wedge q$ — $p$ **y** $q$ (conjunción): verdadera solo si las dos lo son.
- $p \vee q$ — $p$ **o** $q$ (disyunción): verdadera si al menos una lo es.
- $p \rightarrow q$ — **si** $p$ **entonces** $q$ (implicación): solo es falsa cuando $p$ es verdadera
  y $q$ falsa.
- $p \leftrightarrow q$ — $p$ **si y solo si** $q$ (equivalencia): verdadera cuando las dos valen igual.

No vamos a analizar texto con `eval`: representamos cada fórmula como un **árbol** (una estructura de
datos anidada) y escribimos a mano cómo se evalúa. Así controlamos exactamente lo que pasa.

In [ ]:
import itertools

# Una formula es un arbol: ('var','p'), ('not',f), ('and',f,g), ('or',f,g),
# ('imp',f,g), ('iff',f,g). Estos constructores la hacen legible.
def Var(nombre): return ('var', nombre)
def No(a):       return ('not', a)
def Y(a, b):     return ('and', a, b)
def O(a, b):     return ('or', a, b)
def Si(a, b):    return ('imp', a, b)    # a -> b
def Sii(a, b):   return ('iff', a, b)    # a <-> b

def evalua(f, asig):
    """Verdadero/falso de la formula f bajo una asignacion {nombre: bool}."""
    op = f[0]
    if op == 'var': return asig[f[1]]
    if op == 'not': return not evalua(f[1], asig)
    if op == 'and': return evalua(f[1], asig) and evalua(f[2], asig)
    if op == 'or':  return evalua(f[1], asig) or evalua(f[2], asig)
    if op == 'imp': return (not evalua(f[1], asig)) or evalua(f[2], asig)
    if op == 'iff': return evalua(f[1], asig) == evalua(f[2], asig)
    raise ValueError("conectivo desconocido: " + str(op))

_SIMBOLO = {'and': '∧', 'or': '∨', 'imp': '→', 'iff': '↔'}
def texto(f):
    op = f[0]
    if op == 'var': return f[1]
    if op == 'not': return '¬' + texto(f[1])
    return '(' + texto(f[1]) + ' ' + _SIMBOLO[op] + ' ' + texto(f[2]) + ')'

def variables(f):
    op = f[0]
    if op == 'var': return {f[1]}
    if op == 'not': return variables(f[1])
    return variables(f[1]) | variables(f[2])

# Una formula de ejemplo: "si llueve entonces me mojo".
formula = Si(Var('llueve'), Var('me_mojo'))
print("formula:", texto(formula))
print("variables:", sorted(variables(formula)))
print("si llueve=True, me_mojo=False  ->", evalua(formula, {'llueve': True, 'me_mojo': False}))
print("si llueve=True, me_mojo=True   ->", evalua(formula, {'llueve': True, 'me_mojo': True}))

## 2. Tablas de verdad: probar *todos* los mundos posibles

Con $n$ variables hay $2^n$ formas de asignarles verdadero o falso: $2^n$ «mundos posibles». La
**tabla de verdad** los recorre todos y anota qué vale la fórmula en cada uno. Es la fuerza bruta de
la lógica: lenta, pero nunca se equivoca. La construimos generando todas las combinaciones con
`itertools.product`.

In [ ]:
def tabla_de_verdad(f):
    vs = sorted(variables(f))
    filas = []
    for combo in itertools.product([False, True], repeat=len(vs)):
        asig = dict(zip(vs, combo))
        filas.append((asig, evalua(f, asig)))
    return vs, filas

def imprime_tabla(f):
    vs, filas = tabla_de_verdad(f)
    cab = "  ".join(f"{v:>7}" for v in vs) + "  | " + texto(f)
    print(cab)
    print("-" * len(cab))
    for asig, val in filas:
        celdas = "  ".join(f"{('V' if asig[v] else 'F'):>7}" for v in vs)
        print(celdas + "  |  " + ("V" if val else "F"))

imp = Si(Var('p'), Var('q'))
imprime_tabla(imp)
print()
print("La implicacion p -> q solo es FALSA en la fila p=V, q=F.")
print("Numero de filas (mundos posibles):", 2 ** len(variables(imp)))

## 3. Válida, satisfacible o contradicción

Mirando la columna de la derecha de la tabla, clasificamos cualquier fórmula:

- **Válida** (tautología): verdadera en *todos* los mundos. No aporta información, pero es un
  esqueleto seguro del razonamiento; por ejemplo $p \vee \neg p$ («o llueve o no llueve»).
- **Satisfacible** (contingente): verdadera en *algún* mundo. La mayoría de fórmulas útiles.
- **Insatisfacible** (contradicción): falsa en *todos*; por ejemplo $p \wedge \neg p$.

Esta distinción es el motor de todo lo que viene: demostrar que un argumento es correcto se reduce a
demostrar que una cierta fórmula es **insatisfacible**.

In [ ]:
def clasifica(f):
    _, filas = tabla_de_verdad(f)
    valores = [v for _, v in filas]
    if all(valores): return "valida (tautologia)"
    if any(valores): return "satisfacible (contingente)"
    return "insatisfacible (contradiccion)"

ejemplos = [
    O(Var('p'), No(Var('p'))),                 # p v -p
    Y(Var('p'), No(Var('p'))),                 # p ^ -p
    Si(Var('p'), Var('q')),                    # p -> q
    Si(Y(Si(Var('p'), Var('q')), Var('p')), Var('q')),   # ((p->q) ^ p) -> q  (modus ponens)
]
for f in ejemplos:
    print(f"{texto(f):<34} -> {clasifica(f)}")

## 4. Equivalencias: comprobarlas, no creerlas

Dos fórmulas son **equivalentes** si valen lo mismo en *todos* los mundos. Las equivalencias son las
reglas con las que reescribimos una fórmula sin cambiar su significado, igual que en álgebra. Las dos
que más usaremos:

- La **implicación** no es un conectivo primitivo: $p \rightarrow q \equiv \neg p \vee q$.
- Las leyes de **De Morgan**: $\neg(p \wedge q) \equiv \neg p \vee \neg q$.

No hace falta fiarse de la intuición: comparamos las dos fórmulas en todas las asignaciones.

In [ ]:
def equivalentes(f, g):
    vs = sorted(variables(f) | variables(g))
    for combo in itertools.product([False, True], repeat=len(vs)):
        asig = dict(zip(vs, combo))
        if evalua(f, asig) != evalua(g, asig):
            return False
    return True

pruebas = [
    ("implicacion",  Si(Var('p'), Var('q')),               O(No(Var('p')), Var('q'))),
    ("De Morgan",    No(Y(Var('p'), Var('q'))),             O(No(Var('p')), No(Var('q')))),
    ("doble negacion", No(No(Var('p'))),                    Var('p')),
    ("contrapositiva", Si(Var('p'), Var('q')),              Si(No(Var('q')), No(Var('p')))),
    ("falsa a proposito", Si(Var('p'), Var('q')),           Si(Var('q'), Var('p'))),
]
for nombre, f, g in pruebas:
    ok = equivalentes(f, g)
    print(f"{nombre:<18} {texto(f):<16} == {texto(g):<16} ?  {ok}")

## 5. De la fórmula a las cláusulas: la forma normal conjuntiva

Para que una máquina razone con eficiencia conviene poner todas las fórmulas en el mismo molde: la
**forma normal conjuntiva** (CNF), una conjunción («y») de **cláusulas**, donde cada cláusula es una
disyunción («o») de **literales** (una variable o su negación). Toda fórmula tiene una CNF equivalente,
y se obtiene en tres pasos mecánicos:

1. **Eliminar** $\leftrightarrow$ e $\rightarrow$ usando las equivalencias de antes.
2. **Empujar las negaciones** hacia dentro con De Morgan, hasta que solo nieguen variables (forma
   normal negativa).
3. **Distribuir** el «o» sobre el «y»: $a \vee (b \wedge c) \equiv (a \vee b) \wedge (a \vee c)$.

Una cláusula se representa muy bien como un **conjunto de literales**: $\{p, \neg q\}$ es $p \vee \neg q$.
Un literal es un par `(nombre, signo)`, con `signo` verdadero para la variable y falso para su negación.

In [ ]:
def elimina_iff_imp(f):
    op = f[0]
    if op == 'var': return f
    if op == 'not': return No(elimina_iff_imp(f[1]))
    a, b = elimina_iff_imp(f[1]), elimina_iff_imp(f[2])
    if op == 'and': return Y(a, b)
    if op == 'or':  return O(a, b)
    if op == 'imp': return O(No(a), b)                      # a->b  ==  -a v b
    if op == 'iff': return Y(O(No(a), b), O(No(b), a))      # a<->b == (-a v b)^(-b v a)

def a_nnf(f):
    """Empuja negaciones hacia dentro (asume sin -> ni <->)."""
    op = f[0]
    if op == 'var': return f
    if op == 'and': return Y(a_nnf(f[1]), a_nnf(f[2]))
    if op == 'or':  return O(a_nnf(f[1]), a_nnf(f[2]))
    if op == 'not':
        g = f[1]
        if g[0] == 'var': return f
        if g[0] == 'not': return a_nnf(g[1])                       # doble negacion
        if g[0] == 'and': return O(a_nnf(No(g[1])), a_nnf(No(g[2])))  # De Morgan
        if g[0] == 'or':  return Y(a_nnf(No(g[1])), a_nnf(No(g[2])))  # De Morgan
    raise ValueError

def distribuye(f):
    op = f[0]
    if op in ('var', 'not'): return f
    if op == 'and': return Y(distribuye(f[1]), distribuye(f[2]))
    a, b = distribuye(f[1]), distribuye(f[2])
    if a[0] == 'and': return Y(distribuye(O(a[1], b)), distribuye(O(a[2], b)))
    if b[0] == 'and': return Y(distribuye(O(a, b[1])), distribuye(O(a, b[2])))
    return O(a, b)

def literal(f):
    if f[0] == 'var': return (f[1], True)
    if f[0] == 'not' and f[1][0] == 'var': return (f[1][1], False)
    raise ValueError("no es un literal")

def a_clausulas(f):
    """Devuelve un conjunto de clausulas (frozenset de literales) a partir de una CNF."""
    if f[0] == 'and':
        return a_clausulas(f[1]) | a_clausulas(f[2])
    lits = set()
    def recoge(g):
        if g[0] == 'or':
            recoge(g[1]); recoge(g[2])
        else:
            lits.add(literal(g))
    recoge(f)
    return {frozenset(lits)}

def a_cnf(f):
    return a_clausulas(distribuye(a_nnf(elimina_iff_imp(f))))

def lit_texto(lit):
    nombre, signo = lit
    return nombre if signo else '¬' + nombre
def clausula_texto(c):
    if not c: return '□'   # clausula vacia
    return '{' + ', '.join(sorted(lit_texto(l) for l in c)) + '}'

# Ejemplo: "si p entonces (q o no r)" pasada a CNF.
f = Si(Var('p'), O(Var('q'), No(Var('r'))))
print("formula original :", texto(f))
print("sin -> y <->     :", texto(elimina_iff_imp(f)))
print("en NNF           :", texto(a_nnf(elimina_iff_imp(f))))
cnf = a_cnf(f)
print("clausulas (CNF)  :", "  ".join(clausula_texto(c) for c in cnf))

# Un caso donde de verdad hay que distribuir.
g = O(Var('a'), Y(Var('b'), Var('c')))       # a v (b ^ c)
print()
print("formula original :", texto(g))
print("clausulas (CNF)  :", "  ".join(clausula_texto(c) for c in a_cnf(g)))

## 6. Resolución: una sola regla para demostrar

Aquí está la idea central del cuaderno. La **resolución** es una única regla de inferencia que opera
sobre cláusulas. Si una cláusula contiene un literal $L$ y otra contiene su negación $\neg L$, podemos
combinarlas eliminando ese par y juntando el resto:

$$\frac{\{L,\ \alpha_1, \dots\} \qquad \{\neg L,\ \beta_1, \dots\}}{\{\alpha_1, \dots,\ \beta_1, \dots\}}$$

El resultado, el **resolvente**, es una consecuencia lógica de las dos cláusulas de partida. Por
ejemplo, de $\{p, q\}$ (o $p$ o $q$) y $\{\neg p, r\}$ (si $p$, entonces $r$) sale $\{q, r\}$: si no
fuera $q$, tendría que ser $p$, y entonces $r$. Una regla, aplicada una y otra vez, basta para
demostrar cualquier cosa que se siga lógicamente.

In [ ]:
def resolventes(c1, c2):
    """Todos los resolventes de dos clausulas (uno por cada par complementario)."""
    salida = []
    for (nombre, signo) in c1:
        if (nombre, not signo) in c2:
            nueva = (c1 - {(nombre, signo)}) | (c2 - {(nombre, not signo)})
            salida.append((nombre, frozenset(nueva)))
    return salida

c1 = frozenset({('p', True), ('q', True)})     # {p, q}
c2 = frozenset({('p', False), ('r', True)})    # {-p, r}
for pivote, r in resolventes(c1, c2):
    print(f"resolver {clausula_texto(c1)} y {clausula_texto(c2)} sobre '{pivote}'  ->  {clausula_texto(r)}")

## 7. Demostrar por refutación: buscar la contradicción

¿Cómo se demuestra que de unas premisas se sigue una conclusión? Con un truco que tienen en común
casi todos los demostradores automáticos: la **refutación**. En vez de derivar la conclusión hacia
delante, **suponemos que es falsa** y buscamos una contradicción. Si añadir la negación de la
conclusión hace el conjunto insatisfacible, es que la conclusión era forzosa.

En resolución, una contradicción aparece cuando derivamos la **cláusula vacía** $\square$: una
cláusula sin literales, que no puede ser verdadera de ninguna manera. El procedimiento es la
**saturación**: resolver pares una y otra vez, añadiendo cada resolvente nuevo, hasta que salga
$\square$ (insatisfacible) o no haya nada nuevo que derivar (satisfacible).

Lo probamos con un **modus ponens encadenado**: de «$p$», «si $p$ entonces $q$» y «si $q$ entonces
$r$», concluir «$r$». Las cláusulas, con la conclusión negada ($\neg r$), son
$\{p\},\ \{\neg p, q\},\ \{\neg q, r\},\ \{\neg r\}$.

In [ ]:
def refuta(iniciales):
    """Saturacion por resolucion. Devuelve (insatisfacible?, origen) con la genealogia."""
    clausulas = list(dict.fromkeys(iniciales))     # sin duplicados, orden estable
    origen = {c: None for c in clausulas}          # None = premisa
    vacia = frozenset()
    cambiado = True
    while cambiado:
        cambiado = False
        actuales = list(clausulas)
        for i in range(len(actuales)):
            for j in range(i + 1, len(actuales)):
                for pivote, r in resolventes(actuales[i], actuales[j]):
                    # descarta resolventes tautologicos (contienen L y -L)
                    if any((n, not s) in r for (n, s) in r):
                        continue
                    if r in origen:
                        continue
                    origen[r] = (actuales[i], actuales[j], pivote)
                    clausulas.append(r)
                    cambiado = True
                    if r == vacia:
                        return True, origen
    return False, origen

def linaje(origen):
    """Pasos de resolucion que llevan a la clausula vacia, en orden de derivacion."""
    vacia = frozenset()
    if origen.get(vacia) is None:
        return []
    pasos, visto = [], set()
    def rec(c):
        o = origen.get(c)
        if o is None or c in visto:
            return
        a, b, pivote = o
        rec(a); rec(b)
        visto.add(c)
        pasos.append((a, b, pivote, c))
    rec(vacia)
    return pasos

# Modus ponens encadenado: p, p->q, q->r  |=  r ?
premisas = [
    frozenset({('p', True)}),                      # p
    frozenset({('p', False), ('q', True)}),        # -p v q   (p -> q)
    frozenset({('q', False), ('r', True)}),        # -q v r   (q -> r)
]
negar_conclusion = frozenset({('r', False)})       # -r
clausulas = premisas + [negar_conclusion]

print("Cláusulas de partida (conclusión ya negada):")
for c in clausulas:
    print("   ", clausula_texto(c))

insat, origen = refuta(clausulas)
pasos = linaje(origen)
print("\nDerivación por resolución:")
for a, b, pivote, r in pasos:
    print(f"   {clausula_texto(a)} + {clausula_texto(b)}  sobre '{pivote}'  =>  {clausula_texto(r)}")

print("\nLlega a la cláusula vacía:", insat)
print("Conclusión: el argumento es VÁLIDO (r se sigue de las premisas)." if insat
      else "No se deriva contradicción: el argumento NO es válido.")

In [ ]:
# Segundo ejemplo clasico: "si llueve, me mojo" + "llueve"  |=  "me mojo".
lluvia = [
    frozenset({('llueve', False), ('mojado', True)}),   # llueve -> mojado
    frozenset({('llueve', True)}),                      # llueve
    frozenset({('mojado', False)}),                     # negamos la conclusion: -mojado
]
insat, origen = refuta(lluvia)
print("'si llueve me mojo' + 'llueve'  =>  'me mojo' :", "VÁLIDO" if insat else "NO válido")
for a, b, p, r in linaje(origen):
    print(f"   {clausula_texto(a)} + {clausula_texto(b)} sobre '{p}' => {clausula_texto(r)}")

# Contraste: un argumento que NO se sigue. "si llueve me mojo" + "estoy mojado"  =>  "llueve"?
# (afirmar el consecuente: falacia). La saturacion NO debe derivar la clausula vacia.
falacia = [
    frozenset({('llueve', False), ('mojado', True)}),   # llueve -> mojado
    frozenset({('mojado', True)}),                      # mojado
    frozenset({('llueve', False)}),                     # negamos "llueve": -llueve
]
insat2, _ = refuta(falacia)
print("\n'si llueve me mojo' + 'estoy mojado'  =>  'llueve' :", "VÁLIDO" if insat2 else "NO válido")
print("La resolución no encuentra contradicción: estar mojado no demuestra que lloviera.")

## 8. Ver la demostración: el árbol de resolución

La derivación anterior es, en realidad, un **árbol**: las cláusulas de partida arriba, cada resolvente
colgando de las dos cláusulas que lo produjeron, y la cláusula vacía $\square$ al final como raíz de la
contradicción. Dibujarlo deja claro que una demostración no es un acto de fe, sino una cadena de pasos
mecánicos que cualquiera puede verificar.

In [ ]:
import matplotlib.pyplot as plt

insat, origen = refuta(premisas + [negar_conclusion])
pasos = linaje(origen)

fig, ax = plt.subplots(figsize=(8, 0.95 * (len(pasos) + 1)))
ax.axis('off')
y = len(pasos)
for a, b, pivote, r in pasos:
    etiqueta = (f"{clausula_texto(a)}   +   {clausula_texto(b)}"
                f"   --[{pivote}]-->   {clausula_texto(r)}")
    ax.text(0.5, y, etiqueta, ha='center', va='center', family='monospace', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.4', fc='#eef3ff', ec='#3050a0'))
    if y > 1:
        ax.annotate('', xy=(0.5, y - 0.62), xytext=(0.5, y - 0.38),
                    arrowprops=dict(arrowstyle='->', color='#3050a0'))
    y -= 1
ax.set_xlim(0, 1)
ax.set_ylim(0.3, len(pasos) + 0.7)
ax.set_title('Refutación por resolución: cada paso deriva una cláusula nueva\n'
             'hasta la cláusula vacía □ (la contradicción)')
plt.tight_layout()
plt.show()
print("Pasos de resolución hasta la cláusula vacía:", len(pasos))

## 9. SAT y DPLL: decidir sin mirarlo todo

Preguntar «¿es satisfacible esta CNF?» es el famoso problema **SAT**, el primero que se demostró
NP-completo. La tabla de verdad lo resuelve mirando $2^n$ filas, pero eso es inviable en cuanto $n$
crece. El algoritmo **DPLL** (Davis–Putnam–Logemann–Loveland, 1962) es la base de los resolutores SAT
modernos y hace algo más astuto:

- **Propagación unitaria**: si una cláusula tiene un solo literal, ese literal está *obligado*; lo
  fijamos y simplificamos. Esto suele provocar una cascada de obligaciones, gratis.
- **Decisión y backtracking**: cuando ya no queda nada obligado, *elegimos* una variable, probamos un
  valor y, si lleva a contradicción (una cláusula vacía), volvemos atrás y probamos el otro.

Es la misma idea de búsqueda con poda del Facsímil 2 (capítulo 5, los CSP): no explorar las ramas que
ya sabemos condenadas. Contaremos cuántas **decisiones** toma, frente a las $2^n$ filas de la tabla.

In [ ]:
def simplifica(clausulas, lit):
    """Fija el literal lit a verdadero: borra clausulas satisfechas y el literal opuesto."""
    nombre, signo = lit
    opuesto = (nombre, not signo)
    nuevas = []
    for c in clausulas:
        if lit in c:
            continue                      # clausula ya satisfecha
        nuevas.append(c - {opuesto})      # cae el literal que no se puede cumplir
    return nuevas

def dpll(clausulas, cont):
    clausulas = [set(c) for c in clausulas]
    # 1) propagacion unitaria
    while True:
        unitaria = next((c for c in clausulas if len(c) == 1), None)
        if unitaria is None:
            break
        if any(len(c) == 0 for c in clausulas):
            return False
        lit = next(iter(unitaria))
        cont['propagaciones'] += 1
        clausulas = simplifica(clausulas, lit)
    if any(len(c) == 0 for c in clausulas):
        return False                      # conflicto
    if not clausulas:
        return True                       # todas satisfechas
    # 2) decision + backtracking
    nombre = next(iter(clausulas[0]))[0]
    cont['decisiones'] += 1
    if dpll(simplifica(clausulas, (nombre, True)), cont):
        return True
    return dpll(simplifica(clausulas, (nombre, False)), cont)

def es_sat(clausulas):
    cont = {'decisiones': 0, 'propagaciones': 0}
    return dpll(clausulas, cont), cont

# La CNF del modus ponens encadenado SIN negar la conclusion es satisfacible;
# CON la conclusion negada es insatisfacible (por eso el argumento era valido).
sat1, c1 = es_sat([f for f in premisas])
sat2, c2 = es_sat([f for f in premisas] + [negar_conclusion])
print("p, p->q, q->r           ->", "SAT" if sat1 else "UNSAT",
      f"(decisiones={c1['decisiones']}, propagaciones={c1['propagaciones']})")
print("p, p->q, q->r y ademas -r ->", "SAT" if sat2 else "UNSAT",
      f"(decisiones={c2['decisiones']}, propagaciones={c2['propagaciones']})")

In [ ]:
# Comparacion directa con la fuerza bruta en este caso concreto (3 variables: p, q, r).
n_vars = len({nombre for c in (premisas + [negar_conclusion]) for (nombre, _) in c})
print(f"Variables: {n_vars}  ->  la tabla de verdad tiene 2^{n_vars} = {2**n_vars} filas.")
print(f"DPLL decide UNSAT con {c2['decisiones']} decisiones y {c2['propagaciones']} propagaciones unitarias.")
print("Todo queda forzado por propagacion: ni una sola rama que explorar.")

Un caso de tres variables no impresiona. La diferencia se ve cuando $n$ crece: la tabla de verdad
siempre necesita $2^n$ filas, mientras que DPLL, con su propagación y su poda, suele decidir mirando
una fracción minúscula. Generamos fórmulas 3-SAT al azar (con semilla fija, para que el gráfico sea
reproducible) y comparamos, en escala logarítmica, las filas de la fuerza bruta con las decisiones de
DPLL.

In [ ]:
import numpy as np
rng = np.random.default_rng(7)

def cnf_3sat_aleatoria(n, m, rng):
    """m clausulas de 3 literales sobre n variables x0..x(n-1)."""
    clausulas = []
    for _ in range(m):
        idx = rng.choice(n, size=3, replace=False)
        c = set((f"x{i}", bool(rng.integers(0, 2))) for i in idx)
        clausulas.append(frozenset(c))
    return clausulas

ns = list(range(4, 19))
filas_fuerza_bruta = [2 ** n for n in ns]
decisiones_dpll = []
for n in ns:
    cnf = cnf_3sat_aleatoria(n, m=3 * n, rng=rng)   # razon clausulas/variables = 3
    _, cont = es_sat(cnf)
    decisiones_dpll.append(cont['decisiones'])

print("  n  | filas 2^n | decisiones DPLL")
for n, fb, dec in zip(ns, filas_fuerza_bruta, decisiones_dpll):
    print(f"  {n:>2} | {fb:>9} | {dec:>5}")

plt.figure(figsize=(8, 5))
plt.semilogy(ns, filas_fuerza_bruta, 'o-', label='fuerza bruta: $2^n$ filas')
plt.semilogy(ns, [d + 1 for d in decisiones_dpll], 's-',
             label='DPLL: decisiones (+1 para la escala log)')
plt.xlabel('número de variables (n)')
plt.ylabel('trabajo (escala logarítmica)')
plt.title('Fuerza bruta frente a DPLL: por qué la poda lo cambia todo')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nCon n={ns[-1]}: tabla de verdad = {filas_fuerza_bruta[-1]:,} filas; "
      f"DPLL = {decisiones_dpll[-1]} decisiones.")

## 10. Unificación: lógica de primer orden

Hasta aquí, todo eran proposiciones atómicas sin estructura interna. La **lógica de primer orden**
añade lo que faltaba: variables, constantes y funciones dentro de los hechos. «Sócrates es mortal» ya
no es una letra suelta, sino `mortal(socrates)`, y podemos decir «*todos* los humanos son mortales»:
`mortal(X) :- humano(X)`, con $X$ una variable que puede valer cualquier cosa.

Para razonar con esto hace falta una pieza nueva: la **unificación**. Es el algoritmo que busca una
**sustitución** de variables que haga idénticos dos términos. `f(X, b)` y `f(a, Y)` se unifican con
$\{X \mapsto a,\ Y \mapsto b\}$. Hay un detalle delicado, el **occurs-check**: una variable no puede
unificarse con un término que la contenga (no se puede resolver $X = f(X)$), o construiríamos términos
infinitos. Lo implementamos con cuidado.

In [ ]:
# Termino: variable ('var', nombre) o compuesto ('fun', functor, [args]).
# Una constante es un functor sin argumentos.
def V(n):           return ('var', n)
def Cte(n):         return ('fun', n, [])
def Fn(n, *args):   return ('fun', n, list(args))

def es_var(t): return t[0] == 'var'

def camina(t, sust):
    """Sigue la cadena de sustituciones mientras t sea una variable ya ligada."""
    while es_var(t) and t[1] in sust:
        t = sust[t[1]]
    return t

def ocurre(var, t, sust):
    t = camina(t, sust)
    if t == var:
        return True
    if es_var(t):
        return False
    return any(ocurre(var, a, sust) for a in t[2])

def unifica(x, y, sust=None):
    if sust is None:
        sust = {}
    x, y = camina(x, sust), camina(y, sust)
    if x == y:
        return sust
    if es_var(x):
        if ocurre(x, y, sust):           # occurs-check
            return None
        s = dict(sust); s[x[1]] = y; return s
    if es_var(y):
        return unifica(y, x, sust)
    if x[1] != y[1] or len(x[2]) != len(y[2]):   # functores o aridad distintos
        return None
    for a, b in zip(x[2], y[2]):
        sust = unifica(a, b, sust)
        if sust is None:
            return None
    return sust

def aplica(t, sust):
    t = camina(t, sust)
    if es_var(t):
        return t
    return ('fun', t[1], [aplica(a, sust) for a in t[2]])

def term_texto(t):
    if es_var(t):
        return t[1].upper()
    if not t[2]:
        return t[1]
    return t[1] + '(' + ', '.join(term_texto(a) for a in t[2]) + ')'

def sust_texto(sust):
    if not sust:
        return '{}'
    return '{' + ', '.join(f"{k.upper()} -> {term_texto(aplica(('var', k), sust))}"
                           for k in sust) + '}'

casos = [
    (Fn('f', V('x'), Fn('g', V('y'))), Fn('f', Cte('a'), Fn('g', Cte('b')))),  # exito
    (Fn('f', V('x'), V('x')),          Fn('f', Cte('a'), Cte('b'))),           # fallo: x=a y x=b
    (V('x'),                           Fn('f', V('x'))),                       # fallo: occurs-check
    (Fn('p', V('x'), Fn('f', V('x'))), Fn('p', Cte('a'), V('y'))),             # exito, y -> f(a)
]
for x, y in casos:
    s = unifica(x, y)
    if s is None:
        print(f"{term_texto(x):<16} ~ {term_texto(y):<16}  ->  NO unifica")
    else:
        print(f"{term_texto(x):<16} ~ {term_texto(y):<16}  ->  {sust_texto(s)}")

## 11. Un mini-Prolog: encadenar reglas para responder

Con unificación ya podemos construir un motor de inferencia de primer orden en miniatura, un
**mini-Prolog**. La base de conocimiento son **cláusulas de Horn**: hechos (`padre(tom, bob)`) y
reglas (`abuelo(X, Z) :- padre(X, Y), padre(Y, Z)`, que se lee «$X$ es abuelo de $Z$ **si** $X$ es
padre de algún $Y$ y ese $Y$ es padre de $Z$»).

Para responder una consulta, el motor hace **encadenamiento hacia atrás**: toma el objetivo, busca un
hecho o una cabeza de regla que **unifique** con él, y si era una regla, sustituye el objetivo por su
cuerpo y sigue. Renombrando las variables de cada regla en cada uso para que no choquen, y registrando
la traza, vemos exactamente cómo deduce la respuesta. Le preguntamos quién es nieto de Tom.

In [ ]:
# Base de conocimiento: lista de reglas (cabeza, cuerpo). Cuerpo vacio = hecho.
kb = [
    (Fn('padre', Cte('tom'), Cte('bob')), []),
    (Fn('padre', Cte('bob'), Cte('ana')), []),
    (Fn('padre', Cte('bob'), Cte('pat')), []),
    (Fn('padre', Cte('pat'), Cte('eva')), []),
    (Fn('abuelo', V('x'), V('z')),
     [Fn('padre', V('x'), V('y')), Fn('padre', V('y'), V('z'))]),
]

def renombra(termino, sufijo):
    if es_var(termino):
        return ('var', termino[1] + sufijo)
    return ('fun', termino[1], [renombra(a, sufijo) for a in termino[2]])

def resuelve(metas, kb, sust, prof, traza, k):
    if not metas:
        yield sust
        return
    meta = aplica(metas[0], sust)
    traza.append('  ' * prof + 'objetivo: ' + term_texto(meta))
    for cabeza, cuerpo in kb:
        k[0] += 1
        suf = '_' + str(k[0])
        cabeza_r = renombra(cabeza, suf)
        cuerpo_r = [renombra(g, suf) for g in cuerpo]
        s2 = unifica(meta, cabeza_r, dict(sust))
        if s2 is None:
            continue
        if cuerpo_r:
            traza.append('  ' * prof + '  regla ' + term_texto(aplica(cabeza_r, s2)) +
                         ' :- ' + ', '.join(term_texto(aplica(g, s2)) for g in cuerpo_r))
        else:
            traza.append('  ' * prof + '  hecho ' + term_texto(aplica(cabeza_r, s2)))
        yield from resuelve(cuerpo_r + metas[1:], kb, s2, prof + 1, traza, k)

def consulta(meta, kb):
    traza, k = [], [0]
    soluciones = []
    for s in resuelve([meta], kb, {}, 0, traza, k):
        soluciones.append(aplica(meta, s))
    return soluciones, traza

pregunta = Fn('abuelo', Cte('tom'), V('quien'))
soluciones, traza = consulta(pregunta, kb)
print("Consulta:", term_texto(pregunta), "\n")
print("Traza del encadenamiento hacia atrás:")
for linea in traza:
    print("  " + linea)
print("\nRespuestas (¿de quién es abuelo tom?):")
for s in soluciones:
    print("   ", term_texto(s))

## 12. Pruébalo tú

El motor está montado; ahora juega con él. Algunas ideas, de menos a más:

- **Tu propio argumento.** Escribe tres premisas como fórmulas, niégale la conclusión, pásalo todo a
  cláusulas con `a_cnf` y lánzale `refuta`. ¿Es válido? Prueba un razonamiento que *parezca* correcto
  pero no lo sea (como la falacia de afirmar el consecuente) y observa que no deriva $\square$.
- **Equivalencias.** Comprueba con `equivalentes` la otra ley de De Morgan,
  $\neg(p \vee q) \equiv \neg p \wedge \neg q$, o la distributiva.
- **El umbral de la dificultad.** En la 3-SAT aleatoria, cambia la razón `m = 3 * n` por `4 * n` y por
  `4.3 * n`. Cerca de 4.3 cláusulas por variable están los problemas más difíciles: verás dispararse
  las decisiones de DPLL. Es una transición de fase real, muy estudiada.
- **Amplía el mini-Prolog.** Añade una regla recursiva `antepasado(X, Z) :- padre(X, Z)` y
  `antepasado(X, Z) :- padre(X, Y), antepasado(Y, Z)`, y pregunta `antepasado(tom, quien)`. Mira cómo
  la recursión recorre el árbol genealógico en la traza.

## 13. Errores comunes

- **Confundir implicación con equivalencia.** $p \rightarrow q$ no dice nada cuando $p$ es falsa: es
  verdadera «por vacío». De ahí que «estoy mojado» no demuestre «llueve»: la regla solo va en un
  sentido.
- **Saltarse el *occurs-check*.** Muchos Prolog reales lo omiten por velocidad, y entonces unificar
  $X$ con $f(X)$ crea un término infinito que cuelga el programa. Si implementas unificación, decide a
  conciencia si lo incluyes.
- **Olvidar renombrar las variables de las reglas.** Si reutilizas una regla sin variables frescas, la
  $X$ de un uso se mezcla con la $X$ de otro y el motor deduce disparates o no deduce nada.
- **Creer que SAT siendo NP-completo es siempre intratable.** Los resolutores modernos, herederos de
  DPLL, despachan a diario fórmulas con millones de variables. «Peor caso exponencial» no es lo mismo
  que «inútil en la práctica».
- **Generar resolventes tautológicos.** Si no descartas las cláusulas que contienen $L$ y $\neg L$ a
  la vez, la saturación se llena de basura inútil y puede no terminar nunca.
- **Mirar `eval` para parsear lógica.** Tentador y peligroso: mezcla la semántica de Python con la de
  la lógica y abre un agujero de seguridad. Mejor un árbol explícito, como aquí.

## 14. Qué te llevas

- Una fórmula lógica es una **estructura de datos** que sabes evaluar; su **tabla de verdad** decide,
  por fuerza bruta y sin error, si es válida, satisfacible o contradictoria.
- Demostrar que un argumento es correcto equivale a demostrar que negar su conclusión vuelve el
  conjunto **insatisfacible**. La **resolución** lo hace con una sola regla, saturando hasta la
  **cláusula vacía**.
- **DPLL** —propagación unitaria más backtracking— es la misma poda de los CSP del Facsímil 2 aplicada
  a SAT: decide mirando una fracción minúscula de los $2^n$ mundos. Lo viste en el gráfico.
- La **unificación** con *occurs-check* es la pieza que lleva la lógica de las proposiciones sueltas a
  los hechos con estructura, y con ella un **mini-Prolog** responde consultas encadenando reglas.
- Esta IA que *deduce* no compite con la que *aprende*: la complementa. Un modelo de lenguaje propone;
  un motor lógico **verifica**. Buena parte de la IA fiable de los próximos años vive en esa frontera.

**Para seguir:** los capítulos siguientes del facsímil conectan esto con la planificación, con las
bases de conocimiento y con los sistemas neuro-simbólicos que combinan redes y lógica. Y cada vez que
veas «resolutor SAT», «verificación formal» o «motor de reglas», recuerda que por dentro late lo que
acabas de programar.

---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*